# Moirai - Notebook Independiente

Este notebook corre Moirai de forma aislada (sus dependencias `uni2ts`/`gluonts` causan conflictos con torchvision).

**Flujo:**
1. Instalar dependencias + reiniciar runtime
2. Cargar datos (misma lógica que main)
3. Entrenar Moirai + evaluar
4. Guardar predicciones y métricas en Google Drive
5. El notebook principal (`main_forecasting.ipynb`) las carga automáticamente

In [ ]:
# Celda 1: Instalar todo y reiniciar
!git clone https://github.com/allandbb/heartrate-forecasting.git
%cd heartrate-forecasting
!pip install -q -r requirements.txt
!pip install -q uni2ts gluonts einops

# Reiniciar runtime para que numpy/scipy queden consistentes
import os; os.kill(os.getpid(), 9)

In [ ]:
# Celda 2: Posicionar + montar Drive
%cd /content/heartrate-forecasting

from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = '/content/drive/MyDrive/hr_forecasting_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'\U0001f4c1 Caché en: {CACHE_DIR}')

import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Celda 3: Cargar datos (misma lógica que main)
import utils
from sklearn.model_selection import train_test_split

folder_path = 'dataset/'
dfSelectedMean = utils.loadAllFiles(folder_path)

inp = 200
out = 200

df_70, df_30 = utils.selectRandomColumns(dfSelectedMean)

pathEst_70 = 'dataset/values_deses_70.csv'
pathEst_30 = 'dataset/values_deses_30.csv'

df_scaled_70, values_deses_70 = utils.estandarizar(df_70, pathEst_70)
df_scaled_30, values_deses_30 = utils.estandarizar(df_30, pathEst_30)

X_train, y_train, ids_list_tr = utils.series_to_supervised_matrix(
    df_scaled_70.iloc[:, :-2], input_size=inp, output_size=out
)

X_tr, X_va, y_tr, y_va = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

X_test, y_test, ids_list_te = utils.series_to_supervised_matrix(
    df_scaled_30.iloc[:, :-2], input_size=inp, output_size=out
)

print('Datos listos!')
print('Shapes:', X_tr.shape, y_tr.shape, X_test.shape, y_test.shape)

In [ ]:
# Celda 4: Entrenar Moirai + guardar a Drive
import os, json, numpy as np

_moirai_pred_path = os.path.join(CACHE_DIR, 'y_pred_moirai.npy')
_moirai_met_path  = os.path.join(CACHE_DIR, 'metrics_moirai.json')

if os.path.exists(_moirai_pred_path) and os.path.exists(_moirai_met_path):
    print('\u26a1 Cargando Moirai desde Google Drive...')
    y_pred_moirai = np.load(_moirai_pred_path)
    with open(_moirai_met_path) as f:
        metrics_moirai = json.load(f)
    print(f'  Métricas: {metrics_moirai}')
else:
    from wrappers.MoiraiSupervisedWrapper import MoiraiSupervisedWrapper
    moirai_wrapper = MoiraiSupervisedWrapper(config_path='configs/moirai_config.yaml')
    moirai_wrapper.fit(X_tr, y_tr, epochs=20, lr=5e-4)
    metrics_moirai = moirai_wrapper.evaluate(X_test, y_test)
    y_pred_moirai = moirai_wrapper.predict(X_test)

    # Guardar a Google Drive
    np.save(_moirai_pred_path, y_pred_moirai)
    with open(_moirai_met_path, 'w') as f:
        json.dump(metrics_moirai, f)
    print('\U0001f4be Moirai guardado en Google Drive.')

print(f'\nResultados Moirai:')
for k, v in metrics_moirai.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Celda 5: Visualización
utils.plot_forecast_samples(y_test, y_pred_moirai, n_samples=6,
                            title='Moirai: Predicción vs Real')
utils.plot_error_over_horizon(y_test, y_pred_moirai,
                              title='Moirai: Error según horizonte')
utils.plot_subject_performance(y_test, y_pred_moirai, ids_list_te,
                               title='Moirai: Rendimiento por Sujeto')
utils.plot_all_subjects_timeline(y_test, y_pred_moirai, ids_list_te, inp,
                                 title='Moirai: Línea de Tiempo')

---
**Listo.** Las predicciones y métricas están en Google Drive.

Ahora podés ir al `main_forecasting.ipynb` — la celda de Moirai va a cargar los resultados automáticamente desde Drive.